# Trajectory Analysis across models for _Suo et. al._ with Bootstapped Subgraphs

In [1]:
0

0

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import sys
working_directory = "/home/icb/kemal.inecik/work/codes/sctram"
sys.path.append(working_directory)

import logging
import subprocess
import gc
import os
import time
import numpy as np
import pandas as pd
import networkx as nx
import scanpy as sc
import anndata as ad

sc.settings.verbose = 3

In [4]:
from sctram.api._lower_level import TrajectoryEvaluationAPI
from sctram.input import InputTrajectories
from sctram.generate.real import sc_suo_developmental_complete

2025-03-12 16:09:34.738 | INFO     | sctram.api._defaults_read:load_default_metrics:23 - Loaded default metrics from /home/icb/kemal.inecik/work/codes/sctram/sctram/api/_defaults.yaml
2025-03-12 16:09:34.798 | INFO     | sctram.api._defaults_read:load_default_metrics:79 - Default metrics YAML structure validated successfully.


In [5]:
# Important to have consistent figures across platforms

%matplotlib inline
%config InlineBackend.figure_format='retina'

import pickle

from networkx.drawing.nx_agraph import graphviz_layout
from matplotlib import gridspec
import matplotlib.pyplot as plt
import seaborn as sns
import colorcet as cc
from adjustText import adjust_text  
import matplotlib.patheffects as path_effects

_rcparams_path = os.path.join(working_directory, "reproducibility/figure_rcparams/rcparams.pickle")
with open(_rcparams_path, "rb") as file:
    _rcparams = pickle.load(file)
plt.rcParams.update(_rcparams)

Download the dataset and preliminary subsetting

In [6]:
dataset_dir = "/home/icb/kemal.inecik/lustre_workspace/temp_sctram_data"
helpers_directory = os.path.join(os.getcwd(), "helper")
logs_directory = os.path.join(os.getcwd(), "logs")

In [7]:
adata_suo_complete = sc_suo_developmental_complete(dataset_dir=dataset_dir)
display(adata_suo_complete)

2025-03-12 16:09:43.789 | WARNING  | sctram.generate.real._download:download_dataset:105 - File PosixPath('/home/icb/kemal.inecik/lustre_workspace/temp_sctram_data/suo_developmental_complete.h5ad') already exists. Skipping download.


AnnData object with n_obs × n_vars = 841922 × 8192
    obs: 'sample_ID', 'organ', 'age', 'cell_type', 'sex', 'sex_inferred', 'concatenated_integration_covariates', 'integration_donor', 'integration_biological_unit', 'integration_sample_status', 'integration_library_platform_coarse', 'n_genes', '_scvi_batch', '_scvi_labels', 'LVL3', 'LVL2', 'LVL1', 'LVL0'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'rank_genes_groups'
    obsm: 'Unintegrated', 'X_pca', 'harmony', 'invae', 'scanvi', 'scvi', 'tardis_1', 'tardis_2'

In [39]:
itpn_dict = {
    "adata_suo_input_haematopoeitic_lineage_litc_1.pkl": "1", 
    "adata_suo_input_haematopoeitic_lineage_litc_2.pkl": "2", 
    "adata_suo_input_haematopoeitic_lineage.pkl": "manual"
}

litc_dict = dict()
for itp in itpn_dict.keys():
    itp_path = os.path.join(dataset_dir, itp)
    with open(itp_path, "rb") as _file:
        litc_dict[itp] = pickle.load(_file)

# Running the `sctram` package

In [44]:
df = pd.DataFrame()

lineage = "Haematopoeitic_lineage"
lineage_part = lineage.replace("_lineage", "").lower()
for itp, itp_name in itpn_dict.items():

    litc = litc_dict[itp]
    decompose_method = itpn_dict[itp]
    
    for use_rep in adata_suo_complete.obsm.keys():
        if use_rep == "Unintegrated":
            continue

        for trajectory in sorted(litc.graph["trajectories"]):

            if decompose_method != "manual":
                output_file = os.path.join(dataset_dir, f"_metric_adata_suo_bootstrap_method_{decompose_method}_{lineage_part}_{use_rep}_{trajectory}.pickle")
            else:
                output_file = os.path.join(dataset_dir, f"_metric_adata_suo_{lineage_part}_{use_rep}_{trajectory}.pickle")
            
            if not os.path.exists(output_file) or not os.path.isfile(output_file):
                raise ValueError(f"Job {count+1!r} of {lineage!r} with {use_rep!r} of trajectory {trajectory!r} for method {decompose_method!r}")
            else:
                df_trajectory_obsm = pd.read_pickle(output_file)
                df_trajectory_obsm["bootstrapping_method"] = decompose_method if decompose_method == "manual" else f"method_{decompose_method}"
                df_trajectory_obsm["representation"] = use_rep
                df_trajectory_obsm["trajectory"] = trajectory
                df = pd.concat([df, df_trajectory_obsm])

df.reset_index(drop=True, inplace=True)
df['score'] = pd.to_numeric(df['score'], errors='raise')
df.sort_values(by=["bootstrapping_method", "trajectory", "path", "metric", "representation"], inplace=True, ignore_index=True)
df = df[["bootstrapping_method", "trajectory", "path", "metric", "representation", "score"]]

In [45]:
trajectory_classification_dict = {
    'alternative_myeloid': 'myeloid',
    'b_cell_specialization': 'lymphoid',
    'b_cells': 'lymphoid',
    'cd4': 'lymphoid',
    'complete_stem_trajectory': 'other',
    'dendritic': 'myeloid',
    'early_b_cells': 'lymphoid',
    'early_lymphoid': 'lymphoid',
    'early_stem_trajectory': 'other',
    'elp_branching': 'lymphoid',
    'erythroid': 'mem',
    'erythroid_megakaryocyte': 'mem',
    'gmp_branching': 'myeloid',
    'granulocyte_macrophage': 'myeloid',
    'granulocyte_mast': 'myeloid',
    'granulocyte_monocytes': 'myeloid',
    'haematopoeitic_lineage': 'other',
    'icl_nk': 'lymphoid',
    'macrophage_specialization': 'myeloid',
    'megakaryocyte': 'mem',
    'neutrophil': 'myeloid',
    'stem_cells_and_lymphoid_differentiated_cells': 'lymphoid',
    'stem_cells_and_mem': 'mem',
    'stem_cells_and_myeloid_differentiated_cells': 'myeloid',
    't_cell_mid': 'lymphoid',
    't_cell_nkt': 'lymphoid',
    'tissue_macrophages': 'myeloid'
}

metric_direction_dict = {
    # (lower is better)
    'frobenius': 'decreasing',
    'l1_norm': 'decreasing',
    'graph_edit_distance': 'decreasing',
    'spectral_distance': 'decreasing',
    'hamming_distance': 'decreasing',
    'average_shortest_path_difference': 'decreasing',
    'laplacian_spectral_emd': 'decreasing',
    'clustering_coeff_diff': 'decreasing',
    'weisfeiler_lehman_distance': 'decreasing',
    'maximum_common_subgraph_distance': 'decreasing',
    'random_walk_kernel_distance': 'decreasing',
    'persistence_diagram_distance': 'decreasing',
    'mse': 'decreasing',
    'mae': 'decreasing',
    'dtw_distance': 'decreasing',
    'wasserstein_distance_pseudotime': 'decreasing',
    'cdf_kolmogorov_smirnov': 'decreasing',
    'cdf_cramer_von_mises': 'decreasing',
    'sammons_stress': 'decreasing',
    'wasserstein_distance_embedding': 'decreasing',
    'gearys_c_embedding': 'decreasing',
    'gearys_c_pseudotime': 'decreasing',

    # (unsure)
    'normalized_mean_curvature': 'decreasing',
    
    # (higher is better)
    'accuracy': 'increasing',
    'jaccard_similarity': 'increasing',
    'recall': 'increasing',
    'precision': 'increasing',
    'f1_score': 'increasing',
    'permutation_marginalized_ssim': 'increasing',
    'mantel_correlation': 'increasing',
    'gdv_similarity': 'increasing',
    'gin_gnn_similarity': 'increasing',
    'pearson_correlation': 'increasing',
    'spearman_correlation': 'increasing',
    'kendall_correlation': 'increasing',
    'r_squared_with_spline': 'increasing',
    'r_squared': 'increasing',
    'concordance_index': 'increasing',
    'normalized_mutual_information': 'increasing',
    'mutual_information_kde': 'increasing',
    'branch_silhouette_score': 'increasing',
    'embedding_distance_correlation': 'increasing',
    'graph_based_trustworthiness': 'increasing',
    'neighborhood_preservation_score': 'increasing',
    'directionality_preservation': 'increasing',
    'trajectory_cardinality_validation': 'increasing',
    'morans_i_embedding': 'increasing',
    'morans_i_pseudotime': 'increasing'
}
assert set(df["metric"].unique()) == set(metric_direction_dict.keys())
df["trajectory_class"]= [trajectory_classification_dict.get(r["trajectory"], "NA") for _, r in df.iterrows()]
df = df[["bootstrapping_method", "trajectory", "trajectory_class", "path", "metric", "representation", "score"]]

In [46]:
def rank_metrics(df, metric_direction_dict, metric_index_pos, method='min'):
    ranked_df = pd.DataFrame(index=df.index, columns=df.columns, dtype=float)
    for indices, row in df.iterrows():
        
        metric = indices[metric_index_pos]
        direction = metric_direction_dict.get(metric)
        
        if direction not in ('increasing', 'decreasing'):
            raise ValueError            
        
        numeric_row = pd.to_numeric(row, errors='coerce')
        if numeric_row.isna().any():
            ranked_df.loc[indices] = np.nan
            continue
        
        if direction == 'increasing':
            ranks = numeric_row.rank(method='dense', ascending=False)
        else:
            ranks = numeric_row.rank(method='dense', ascending=True)

        # ranks = 1 - pd.Series(MinMaxScaler().fit_transform(row.values.reshape(-1, 1)).flatten(), index=row.index)
        
        ranked_df.loc[indices] = ranks.astype('Float64')
    return ranked_df

def rank_df(df, direction):
    ranked_df = pd.DataFrame(index=df.index, columns=df.columns, dtype=int)
    for indices, row in df.iterrows():
        numeric_row = pd.to_numeric(row, errors='raise')
        if direction == 'increasing':
            ranks = numeric_row.rank(method='dense', ascending=False)
        else:
            ranks = numeric_row.rank(method='dense', ascending=True)
        ranked_df.loc[indices] = ranks.astype('Int64')
    return ranked_df
    
def min_max_scaler(df):
    return pd.DataFrame(
        MinMaxScaler().fit_transform(df),
        columns=df.columns,
        index=df.index,
    )

def min_max_scaler_rowwise(df):
    def scale_row(row):
        if row.isna().any():  # If any NaN is present, return NaNs
            return pd.Series([np.nan] * len(row), index=row.index)
        return pd.Series(MinMaxScaler().fit_transform(row.values.reshape(-1, 1)).flatten(), index=row.index)
    return df.apply(scale_row, axis=1)

def min_max_scaler_rowwise_direction_aware(df, metric_direction_dict, metric_index_pos):
    ranked_df = pd.DataFrame(index=df.index, columns=df.columns, dtype=float)
    for indices, row in df.iterrows():
        metric = indices[metric_index_pos]
        
        direction = metric_direction_dict.get(metric)
        if direction not in ('increasing', 'decreasing'):
            raise ValueError
        
        numeric_row = pd.to_numeric(row, errors='coerce')
        if numeric_row.isna().any():
            ranked_df.loc[indices] = np.nan
            continue
        
        norm_row = pd.Series(MinMaxScaler().fit_transform(numeric_row.values.reshape(-1, 1)).flatten(), index=numeric_row.index)
        if direction == 'increasing':
            adjusted_row = norm_row
        else:
            adjusted_row = 1 - norm_row
        ranked_df.loc[indices] = adjusted_row.astype('Float64')
    return ranked_df

## Analysis

In [48]:
df

,bootstrapping_method,trajectory,trajectory_class,path,metric,representation,score
0,manual,alternative_myeloid,myeloid,adjacency,accuracy,X_pca,0.920000
1,manual,alternative_myeloid,myeloid,adjacency,accuracy,harmony,0.920000
2,manual,alternative_myeloid,myeloid,adjacency,accuracy,invae,0.920000
3,manual,alternative_myeloid,myeloid,adjacency,accuracy,scanvi,0.760000
4,manual,alternative_myeloid,myeloid,adjacency,accuracy,scvi,0.920000
...,...,...,...,...,...,...,...
56443,method_2,haematopoeitic_lineage_subgraph_99,NA,pseudotime,wasserstein_distance_pseudotime,invae,0.335305
56444,method_2,haematopoeitic_lineage_subgraph_99,NA,pseudotime,wasserstein_distance_pseudotime,scanvi,0.206439
56445,method_2,haematopoeitic_lineage_subgraph_99,NA,pseudotime,wasserstein_distance_pseudotime,scvi,0.360832
56446,method_2,haematopoeitic_lineage_subgraph_99,NA,pseudotime,wasserstein_distance_pseudotime,tardis_1,0.301917


In [ ]:
# df_edges = []
# for ind, row in df.iterrows():
#     trj = lit.get_trajectory(row["trajectory"], False)
#     for from_cell, to_cell in trj.edges():
#         row_copy = row.copy()
#         row_copy["from_cell"] = from_cell
#         row_copy["to_cell"] = to_cell
#         df_edges.append(row_copy)
# df_edges = pd.DataFrame(df_edges)
# df_edges.reset_index(drop=True, inplace=True)

In [ ]:
# df_edges_pivot = df_edges.pivot(index=['path', 'trajectory', 'trajectory_class', 'metric', 'from_cell', 'to_cell'], columns='representation', values='score')
# df_edges_pivot_rank = rank_metrics(df_edges_pivot, metric_direction_dict, metric_index_pos=3, method='min')
# # df_edges_pivot_rank = min_max_scaler_rowwise_direction_aware(df_edges_pivot, metric_direction_dict, metric_index_pos=2)

# df_edges_pivot_rank_mean = df_edges_pivot_rank.groupby(level=[4, 5]).mean()
# df_edges_pivot_rank_mean

In [ ]:
# trj = lit.get_trajectory("haematopoeitic_lineage", False).copy()
# for (from_cell, to_cell), representations in df_edges_pivot_rank_mean.iterrows():
#     for representation, score in representations.items():
#         trj[from_cell][to_cell][f"weight_{representation}"] = score
# list(trj[from_cell][to_cell].keys())

In [ ]:
# trj.plot_trajectory(
#     title="Haematopoeitic Lineage with `scanvi` Edge Weights",
#     figsize=(18, 18), 
#     label_offset=15,
#     color_edges_by_weight=True,
#     edge_weight_attribute="weight_scanvi",
#     edge_labels=True,
#     edge_colorbar=True,
#     # edge_vmin=1,
#     # edge_vmax=4,
#     edge_cmap='viridis'
# )